# Exercise 4: Unified Multi-Agent Workflow

The capstone: compose agents from **two different frameworks** into a single workflow:

```
User Query
    |
    v
+-------------------+
| Orchestrator      |  (Bielik - react_agent)
+---+----------+----+
    |          |
    v          v
+--------+ +----------+
|Research| | Writing  |
|Agent   | | Crew     |
|LangChn | | CrewAI   |
+--------+ +----------+
```

In [1]:
import os
import sys
from dotenv import load_dotenv

load_dotenv()

# Ensure function modules are importable
sys.path.insert(0, os.path.join(os.getcwd(), "..", "02_langchain_agent"))
sys.path.insert(0, os.path.join(os.getcwd(), "..", "03_crewai_agent"))

## Step 1: Review the Unified Config

Both `research_agent` and `writing_crew` listed under `functions:`. Bielik orchestrates both.

In [2]:
with open("config.yaml") as f:
    print(f.read())

# =============================================================
# Exercise 4: Unified Multi-Agent Workflow
# =============================================================
# This is the capstone exercise for Part 2.
#
# It composes TWO agents from different frameworks:
#   1. LangChain ReAct research agent (from Exercise 2)
#   2. CrewAI writing crew (from Exercise 3)
#
# Workflow: User query -> Research Agent -> Writing Crew -> Report
# =============================================================

llms:
  bielik:
    _type: openai
    base_url: "${VLLM_BASE_URL}"
    model: "${VLLM_MODEL_NAME}"
    api_key: "${VLLM_API_KEY}"
    temperature: 0.7
    max_tokens: 2048

functions:
  # --- Tools for the research agent ---
  wiki_search:
    _type: wiki_search
    max_results: 2

  current_time:
    _type: current_datetime

  # --- LangChain research agent (from Exercise 2) ---
  research_agent:
    _type: langchain_research_agent
    llm_ref: bielik
    tools_ref:
      - wiki_search
    

## Step 2: Run the Unified Workflow

In [3]:
!nat run --config_file config.yaml --input "Zbadaj Kopalnię Soli w Wieliczce - jej historię i status UNESCO. Następnie napisz raport."

2026-04-13 18:26:41 - INFO     - nat.cli.commands.start:192 - Starting NAT from config file: 'config.yaml'
2026-04-13 18:26:41 - ERROR    - nat.data_models.config:118 - Requested functions type `langchain_research_agent` not found. Have you ensured the necessary package has been installed with `uv pip install`?
Available functions names:
 - nat.plugins.langchain/langgraph_wrapper
 - nat.plugins.langchain.tools/code_generation
 - nat.plugins.langchain.tools/tavily_internet_search
 - nat.plugins.langchain.tools/wiki_search
 - nat.plugins.langchain.agent.auto_memory_wrapper/auto_memory_agent
 - nat.plugins.langchain.agent.prompt_optimizer/prompt_init
 - nat.plugins.langchain.agent.prompt_optimizer/prompt_recombiner
 - nat.plugins.langchain.agent.react_agent/react_agent
 - nat.plugins.langchain.agent.react_agent/per_user_react_agent
 - nat.plugins.langchain.agent.reasoning_agent/reasoning_agent
 - nat.plugins.langchain.agent.responses_api_agent/responses_api_agent
 - nat.plugins.langchain.

## Step 3: Deploy as an API (Reference)

NAT can serve the entire multi-agent workflow as a REST API:

```bash
nat serve --config_file config.yaml --port 8080
```

Endpoints:
- `POST /generate` - Synchronous result
- `POST /generate/stream` - Streaming result  
- `POST /chat` - OpenAI-compatible
- `GET /health` - Health check

The `nat serve` command starts a web server that listens for incoming requests and runs the specified workflow. The server can be accessed with a web browser or by sending a POST request to the server’s endpoint. 

You can curl the endpoint like so:
```bash
curl --request POST \
  --url http://localhost:8000/generate \
  --header 'Content-Type: application/json' \
  --data '{
    "input_message": "Co to jest Bielik LLM?"
}'
```

In [4]:
# Your turn - try your own topic
!nat run --config_file config.yaml --input "TWÓJ TEMAT - Zbadaj go i napisz raport"

2026-04-13 18:26:43 - INFO     - nat.cli.commands.start:192 - Starting NAT from config file: 'config.yaml'
2026-04-13 18:26:43 - ERROR    - nat.data_models.config:118 - Requested functions type `langchain_research_agent` not found. Have you ensured the necessary package has been installed with `uv pip install`?
Available functions names:
 - nat.plugins.langchain/langgraph_wrapper
 - nat.plugins.langchain.tools/code_generation
 - nat.plugins.langchain.tools/tavily_internet_search
 - nat.plugins.langchain.tools/wiki_search
 - nat.plugins.langchain.agent.auto_memory_wrapper/auto_memory_agent
 - nat.plugins.langchain.agent.prompt_optimizer/prompt_init
 - nat.plugins.langchain.agent.prompt_optimizer/prompt_recombiner
 - nat.plugins.langchain.agent.react_agent/react_agent
 - nat.plugins.langchain.agent.react_agent/per_user_react_agent
 - nat.plugins.langchain.agent.reasoning_agent/reasoning_agent
 - nat.plugins.langchain.agent.responses_api_agent/responses_api_agent
 - nat.plugins.langchain.

## Part 2 Summary

1. **Basic agent** - ReAct agent with Bielik using NAT's built-in tools
2. **LangChain integration** - ReAct research agent registered as NAT Function
3. **CrewAI integration** - Multi-agent crew registered as NAT Function
4. **Unified workflow** - Both frameworks composed via YAML config

**NAT is framework-agnostic.** Write agents in whatever framework you prefer, register them as Functions, compose declaratively.

---

**After the break:** Part 3 - observability, cost tracking, and optimization.